In [0]:
from pyspark.sql import functions as F

def normalize_columns(df):
    """Standardizes column names: lowercase, spaces to underscores."""
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))
    return df

In [0]:
df_order = spark.table("workspace.bronze.order")
df_order = normalize_columns(df_order)

df_order = (
    df_order
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("store_id", F.col("store_id").cast("int"))
    .dropDuplicates()
)

df_order.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.silver.order")
print(f"silver.order: {df_order.count()} rows written")